In [1]:
# =============================================================================
# Intermediate RAG System - Phase 1: Environment Setup
# This is an earlier version using TF-IDF sparse retrieval.
# =============================================================================

import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

def upgrade(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", package])

# core ML stack
upgrade("accelerate")
upgrade("transformers")
upgrade("peft")
upgrade("bitsandbytes")

# sparse retrieval uses sklearn TF-IDF instead of FAISS
install("scikit-learn")

# dataset loading
install("datasets==2.19.2")

# utilities
install("nltk==3.8.1")
install("pandas==2.2.2")
install("tqdm==4.66.4")
install("rouge-score==0.1.2")

print("Installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 13.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2024.3.1 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 49.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
textblob 0.19.0 requires nltk>=3.9, but you have nltk 3.8.1 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 93.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 5.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
dataproc-spark-connect 1.0.2 requires tqdm>=4.67, but you have tqdm 4.66.4 which is incompatible.
textblob 0.19.0 requires nltk>=3.9, but you have nltk 3.8.1 which is incompatible.
tobler 0.13.0 requires tqdm>=4.67, but you have tqdm 4.66.4 which is incompatible.


Installation complete.


In [2]:
# seed and directory setup
import os, sys, random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.makedirs("/kaggle/working/hf_cache", exist_ok=True)
os.makedirs("/kaggle/working/outputs_v1", exist_ok=True)

print("Seed:", SEED)
print("Output directory: /kaggle/working/outputs_v1")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Seed: 42
Output directory: /kaggle/working/outputs_v1
GPU: Tesla T4


In [3]:
# =============================================================================
#  RAG System - Phase 2: Dataset Construction
# Loads MITRE ATT&CK data b
# =============================================================================

import json
import requests
import random
import re
import pandas as pd
from tqdm import tqdm

OUTPUT_PATH = "/kaggle/working/outputs_v1/cybersecurity_kb_v1.jsonl"

MITRE_URL = (
    "https://raw.githubusercontent.com/mitre/cti/master/"
    "enterprise-attack/enterprise-attack.json"
)

print("Downloading MITRE ATT&CK dataset...")
resp = requests.get(MITRE_URL, timeout=60)
resp.raise_for_status()
mitre_data = resp.json()
print(f"Total MITRE objects: {len(mitre_data['objects'])}")

Total MITRE objects: 25842


In [4]:
# text cleaning func clean_text(text):
    text = re.sub(r'\(Citation:[^)]+\)', '', str(text))
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# parse only attack-pattern objects from STIX data
techniques = []
for obj in mitre_data['objects']:
    if obj.get('type') != 'attack-pattern':
        continue
    if obj.get('x_mitre_deprecated', False) or obj.get('revoked', False):
        continue

    name = obj.get('name', '')
    desc = clean_text(obj.get('description', ''))
    mitre_id = next(
        (r['external_id'] for r in obj.get('external_references', [])
         if r.get('source_name') == 'mitre-attack'), ''
    )

    if name and desc:
        techniques.append({
            'name': name,
            'id': mitre_id,
            'description': desc
        })

print(f"Valid techniques parsed: {len(techniques)}")

Valid techniques parsed: 697


In [5]:
# generate simple Q&A pairs 
def make_simple_qa(tech):
    pairs = []
    name = tech['name']
    mid  = tech['id']

    # basic what-is question only
    if tech['description']:
        pairs.append({
            "question": f"What is {name} in cybersecurity?",
            "answer":   tech['description'][:600],
            "domain":   "mitre_attack",
            "source":   f"MITRE ATT&CK {mid}",
            "difficulty": "intermediate"
        })

    # basic how-to-defend question using same description
    if len(tech['description']) > 100:
        pairs.append({
            "question": f"How should defenders respond to {name}?",
            "answer":   tech['description'][:600],
            "domain":   "mitre_attack",
            "source":   f"MITRE ATT&CK {mid}",
            "difficulty": "basic"
        })

    return pairs

qa_pairs = []
for tech in tqdm(techniques, desc="Generating Q&A pairs"):
    qa_pairs.extend(make_simple_qa(tech))

print(f"Total Q&A pairs before sampling: {len(qa_pairs)}")

Generating Q&A pairs: 100%|██████████| 697/697 [00:00<00:00, 291555.79it/s]

Total Q&A pairs before sampling: 1394


In [6]:

random.shuffle(qa_pairs)
qa_subset = qa_pairs[:500]

print(f"Subset size: {len(qa_subset)}")
print(f"Note: limiting to 500 samples reduces domain coverage significantly")

# save to jsonl
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for record in qa_subset:
        f.write(json.dumps(record) + '\n')

print(f"Dataset saved to: {OUTPUT_PATH}")

Subset size: 500
Note: limiting to 500 samples reduces domain coverage significantly
Dataset saved to: /kaggle/working/outputs_v1/cybersecurity_kb_v1.jsonl


In [7]:
# dataset summary
df = pd.DataFrame(qa_subset)
print("DATASET SUMMARY")
print(f"Total samples : {len(df)}")
print()
print("By difficulty:")
print(df['difficulty'].value_counts().to_string())
print()
sample = df.sample(1, random_state=SEED).iloc[0]
print("Sample record:")
print(f"  Source   : {sample['source']}")
print(f"  Question : {sample['question']}")
print(f"  Answer   : {sample['answer'][:120]}...")

DATASET SUMMARY
Total samples : 500

By difficulty:
difficulty
intermediate    251
basic           249

Sample record:
  Source   : MITRE ATT&CK T1068
  Question : What is Exploitation for Privilege Escalation in cybersecurity?
  Answer   : Adversaries may exploit software vulnerabilities in an attempt to elevate privileges. Exploitation of a software vulnera...


In [9]:
# =============================================================================
#  chunking 
# =============================================================================

import json
import re
import pandas as pd
from tqdm import tqdm

INPUT_PATH  = "/kaggle/working/outputs_v1/cybersecurity_kb_v1.jsonl"
OUTPUT_PATH = "/kaggle/working/outputs_v1/chunks_v1.jsonl"

# simple character-based chunk size instead of token-based
# this is less precise than tiktoken used in the final version
CHUNK_SIZE = 800  # characters per chunk

In [10]:
# load dataset
records = []
with open(INPUT_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line.strip()))

print(f"Records loaded: {len(records)}")

Records loaded: 500


In [11]:
# basic text cleaning without HTML or citation removal
# less thorough than the final version
def basic_clean(text):
    text = re.sub(r'\s+', ' ', str(text)).strip()
    return text

# simple character-based chunker with no overlap
# limitation: sentences can be cut mid-way at chunk boundaries
def simple_chunk(text, chunk_size=CHUNK_SIZE):
    chunks = []
    start  = 0
    while start < len(text):
        end        = min(start + chunk_size, len(text))
        chunk_text = text[start:end].strip()
        if chunk_text:
            chunks.append(chunk_text)
        start = end   # no overlap - this is the key limitation
    return chunks

In [12]:
# build documents and chunk them
all_chunks = []
chunk_id   = 0

for doc_idx, r in enumerate(tqdm(records, desc="Chunking")):
    # simple concatenation without Question/Answer prefix labels
    # final version uses "Question: ... Answer: ..." format for richer context
    doc_text = basic_clean(r['question']) + " " + basic_clean(r['answer'])
    doc_chunks = simple_chunk(doc_text)

    for c_idx, chunk_text in enumerate(doc_chunks):
        all_chunks.append({
            "chunk_id":   f"doc{doc_idx}_chunk{c_idx}",
            "doc_id":     doc_idx,
            "text":       chunk_text,
            "char_count": len(chunk_text),
            "source":     r.get("source", ""),
            "domain":     r.get("domain", ""),
            "question":   r.get("question", ""),
            "answer":     r.get("answer", "")
        })
        chunk_id += 1

print(f"Total chunks produced: {len(all_chunks)}")

Chunking: 100%|██████████| 500/500 [00:00<00:00, 10973.30it/s]

Total chunks produced: 500


In [13]:
# save chunks
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk) + '\n')

print(f"Chunks saved to: {OUTPUT_PATH}")

Chunks saved to: /kaggle/working/outputs_v1/chunks_v1.jsonl


In [14]:
# chunking summary
df_chunks = pd.DataFrame(all_chunks)

print("CHUNKING SUMMARY")
print(f"Total chunks        : {len(df_chunks)}")
print(f"Unique source docs  : {df_chunks['doc_id'].nunique()}")
print(f"Avg chars per chunk : {df_chunks['char_count'].mean():.1f}")
print(f"Min chars per chunk : {df_chunks['char_count'].min()}")
print(f"Max chars per chunk : {df_chunks['char_count'].max()}")
print()
print("Note: no overlap between chunks - context loss possible at boundaries")
print()
sample = df_chunks.sample(1, random_state=42).iloc[0]
print("Sample chunk:")
print(f"  chunk_id   : {sample['chunk_id']}")
print(f"  char_count : {sample['char_count']}")
print(f"  source     : {sample['source']}")
print(f"  text       : {sample['text'][:150]}...")

CHUNKING SUMMARY
Total chunks        : 500
Unique source docs  : 500
Avg chars per chunk : 636.4
Min chars per chunk : 262
Max chars per chunk : 687

Note: no overlap between chunks - context loss possible at boundaries

Sample chunk:
  chunk_id   : doc361_chunk0
  char_count : 664
  source     : MITRE ATT&CK T1068
  text       : What is Exploitation for Privilege Escalation in cybersecurity? Adversaries may exploit software vulnerabilities in an attempt to elevate privileges. ...


In [15]:
# =============================================================================
# Section 4: Embedding Generation and Index Construction
# Uses TF-IDF vectorization with cosine similarity for retrieval.
# TF-IDF chosen for simplicity and interpretability at this corpus scale.
# =============================================================================

import json
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

CHUNKS_PATH = "/kaggle/working/outputs_v1/chunks_v1.jsonl"
INDEX_PATH  = "/kaggle/working/outputs_v1/tfidf_index.pkl"

# load chunks
chunks = []
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        chunks.append(json.loads(line.strip()))

texts    = [c["text"] for c in chunks]
metadata = [{k: v for k, v in c.items() if k != "text"} for c in chunks]

print(f"Chunks loaded: {len(chunks)}")

Chunks loaded: 500


In [17]:
# build TF-IDF matrix over the full corpus
# sublinear_tf dampens the effect of very frequent terms
# ngram_range (1,2) captures bigrams for better phrase matching
vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    ngram_range=(1, 2),
    max_features=20000,
    stop_words='english'
)

print("Fitting TF-IDF vectorizer...")
tfidf_matrix = vectorizer.fit_transform(texts)

print(f"TF-IDF matrix shape  : {tfidf_matrix.shape}")
print(f"Vocabulary size      : {len(vectorizer.vocabulary_)}")
print(f"Matrix sparsity      : {1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.4f}")

Fitting TF-IDF vectorizer...
TF-IDF matrix shape  : (500, 18990)
Vocabulary size      : 18990
Matrix sparsity      : 0.9949


In [18]:
# save vectorizer and matrix to disk
with open(INDEX_PATH, 'wb') as f:
    pickle.dump({
        "vectorizer":   vectorizer,
        "tfidf_matrix": tfidf_matrix,
        "metadata":     metadata,
        "texts":        texts
    }, f)

print(f"TF-IDF index saved to: {INDEX_PATH}")

TF-IDF index saved to: /kaggle/working/outputs_v1/tfidf_index.pkl


In [19]:
# retriever using TF-IDF cosine similarity
class TFIDFRetriever:
    # sparse retrieval using TF-IDF weighted term vectors

    def __init__(self, vectorizer, tfidf_matrix, chunks, metadata, top_k=3):
        self.vectorizer   = vectorizer
        self.tfidf_matrix = tfidf_matrix
        self.chunks       = chunks
        self.metadata     = metadata
        self.top_k        = top_k

    def retrieve(self, query, top_k=None):
        k = top_k if top_k else self.top_k
        # transform query into TF-IDF space
        query_vec = self.vectorizer.transform([query])
        scores    = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        top_ids   = np.argsort(scores)[::-1][:k]

        results = []
        for rid in top_ids:
            results.append({
                "chunk_id": self.metadata[rid]["chunk_id"],
                "score":    float(scores[rid]),
                "text":     self.chunks[rid]["text"],
                "source":   self.metadata[rid]["source"],
                "domain":   self.metadata[rid]["domain"],
                "question": self.metadata[rid]["question"],
                "answer":   self.metadata[rid]["answer"]
            })
        return results

retriever = TFIDFRetriever(
    vectorizer   = vectorizer,
    tfidf_matrix = tfidf_matrix,
    chunks       = chunks,
    metadata     = metadata,
    top_k        = 3
)

print("TF-IDF retriever ready.")

TF-IDF retriever ready.


In [20]:
# quick sanity check on retrieval
test_query   = "How can an attacker use phishing to gain initial access?"
test_results = retriever.retrieve(test_query)

print(f"Test query: {test_query}")
print()
for i, r in enumerate(test_results, 1):
    print(f"  Rank {i} | Score: {r['score']:.4f} | Source: {r['source']}")
    print(f"           | Text: {r['text'][:120]}...")
    print()

Test query: How can an attacker use phishing to gain initial access?

  Rank 1 | Score: 0.1566 | Source: MITRE ATT&CK T1204
           | Text: What is User Execution in cybersecurity? An adversary may rely upon specific actions by a user in order to gain executio...

  Rank 2 | Score: 0.1125 | Source: MITRE ATT&CK T1078
           | Text: How should defenders respond to Valid Accounts? Adversaries may obtain and abuse credentials of existing accounts as a m...

  Rank 3 | Score: 0.1115 | Source: MITRE ATT&CK T1078.002
           | Text: What is Domain Accounts in cybersecurity? Adversaries may obtain and abuse credentials of a domain account as a means of...



In [21]:
# save retriever index reference for Section 5
with open("/kaggle/working/outputs_v1/retriever_ready.txt", 'w') as f:
    f.write("TF-IDF retriever constructed successfully.\n")
    f.write(f"Corpus size: {len(chunks)} chunks\n")
    f.write(f"Vocabulary: {len(vectorizer.vocabulary_)} terms\n")

print("Index construction complete.")

Index construction complete.


In [22]:
# =============================================================================
# Section 5: Retrieval Module and Quality Evaluation
# Evaluates TF-IDF retrieval using Hit Rate and MRR across 10 queries.
# =============================================================================

import numpy as np
import pandas as pd
import pickle

# evaluation queries covering key cybersecurity domains
EVAL_QUERIES = [
    {
        "query": "How can an attacker use phishing to gain initial access to a network?",
        "keywords": ["phishing", "spearphishing", "email", "link", "attachment"]
    },
    {
        "query": "What techniques do adversaries use to escalate privileges on Windows?",
        "keywords": ["privilege", "escalat", "windows", "admin", "token", "bypass"]
    },
    {
        "query": "How do attackers move laterally across a compromised network?",
        "keywords": ["lateral", "movement", "remote", "pass-the-hash", "smb", "rdp"]
    },
    {
        "query": "What methods are used to exfiltrate sensitive data from a victim system?",
        "keywords": ["exfiltrat", "data", "transfer", "upload", "channel", "encrypt"]
    },
    {
        "query": "How do adversaries maintain persistence after compromising a system?",
        "keywords": ["persist", "backdoor", "startup", "registry", "scheduled", "boot"]
    },
    {
        "query": "What techniques are used for credential dumping and password theft?",
        "keywords": ["credential", "dump", "password", "lsass", "hash", "mimikatz"]
    },
    {
        "query": "How do attackers evade antivirus and endpoint detection tools?",
        "keywords": ["evasion", "bypass", "antivirus", "obfuscat", "detection", "defence"]
    },
    {
        "query": "What command and control techniques do malware authors use?",
        "keywords": ["command", "control", "c2", "beacon", "callback", "protocol"]
    },
    {
        "query": "How can attackers exploit vulnerabilities in web applications?",
        "keywords": ["exploit", "web", "injection", "vulnerability", "application", "server"]
    },
    {
        "query": "What techniques are used to perform reconnaissance before an attack?",
        "keywords": ["reconnaissance", "scan", "discover", "enumerat", "gather", "osint"]
    },
]

print(f"Evaluation queries: {len(EVAL_QUERIES)}")

Evaluation queries: 10


In [23]:
# relevance check using keyword matching
def is_relevant(text, keywords):
    text_lower = text.lower()
    return any(kw.lower() in text_lower for kw in keywords)

def reciprocal_rank(results, keywords):
    for rank, r in enumerate(results, start=1):
        if is_relevant(r["text"], keywords):
            return 1.0 / rank
    return 0.0

eval_records = []

print("Running retrieval evaluation.")
print()

for eq in EVAL_QUERIES:
    results  = retriever.retrieve(eq["query"], top_k=3)
    hit      = any(is_relevant(r["text"], eq["keywords"]) for r in results)
    rr       = reciprocal_rank(results, eq["keywords"])
    top_score= results[0]["score"] if results else 0.0

    eval_records.append({
        "query":      eq["query"],
        "hit_at_3":   int(hit),
        "rr":         rr,
        "top_score":  top_score,
        "top_source": results[0]["source"] if results else ""
    })

df_eval = pd.DataFrame(eval_records)

Running retrieval evaluation...



In [30]:
# per query results table
print("PER-QUERY RETRIEVAL RESULTS")
print()
print(f"{'Query':<55} {'Hit':>4} {'RR':>6} {'Score':>7}")
print("-" * 80)
for _, row in df_eval.iterrows():
    print(
        f"{row['query'][:53]:<55} {row['hit_at_3']:>4} "
        f"{row['rr']:>6.2f} {row['top_score']:>7.4f}"
    )
print("-" * 80)
print()

hit_rate = df_eval["hit_at_3"].mean()
mrr      = df_eval["rr"].mean()
avg_score= df_eval["top_score"].mean()

print("AGGREGATE RETRIEVAL METRICS")
print(f"  Hit Rate @ 3   : {hit_rate:.4f}  ({int(hit_rate*10)}/10 queries)")
print(f"  MRR @ 3        : {mrr:.4f}")
print(f"  Avg Top-1 Score: {avg_score:.4f}")
print()
print("DIAGNOSIS")
if hit_rate < 0.5:
    print("  Hit rate below 0.5 indicates TF-IDF retrieval is insufficient")
    print("  for this cybersecurity domain due to vocabulary mismatch.")
    print("  Queries using semantic phrasing fail to match technique descriptions")
    print("  which use different but related terminology.")
    print("  This motivates switching to dense embedding-based retrieval.")
if avg_score < 0.3:
    print("  between query terms and corpus vocabulary.")

PER-QUERY RETRIEVAL RESULTS

Query                                                    Hit     RR   Score
--------------------------------------------------------------------------------
How can an attacker use phishing to gain initial acce      1   0.50  0.1741
What techniques do adversaries use to escalate privil      1   1.00  0.0995
How do attackers move laterally across a compromised       1   1.00  0.1617
What methods are used to exfiltrate sensitive data fr      1   1.00  0.1269
How do adversaries maintain persistence after comprom      1   1.00  0.1291
What techniques are used for credential dumping and p      1   1.00  0.1984
How do attackers evade antivirus and endpoint detecti      1   1.00  0.2144
What command and control techniques do malware author      1   1.00  0.2006
How can attackers exploit vulnerabilities in web appl      1   1.00  0.1916
What techniques are used to perform reconnaissance be      0   0.00  0.1487
------------------------------------------------------

In [27]:
# save evaluation records for comparison with final version
import json

with open("/kaggle/working/outputs_v1/retrieval_eval_v1.json", 'w') as f:
    json.dump(eval_records, f, indent=2)

print()
print("Retrieval evaluation saved.")
print()



Retrieval evaluation saved.

